<!-- NB_DOC_INTRO_V1 -->
# Modelisation Bayesienne

> 📚 **Doc thematique** : [docs/03_ML.md](docs/03_ML.md) (Machine Learning classique)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**Statistiques frequentistes** : "Les parametres sont fixes, les donnees sont aleatoires."
**Statistiques bayesiennes** : "Les donnees sont fixes, les parametres sont des distributions."

Ce notebook execute des analyses bayesiennes **avec scipy** (pour les cas conjugues simples, exhaustivement) et presente **PyMC** (le standard) en code pret a executer.

**Avantages de l'approche bayesienne** :
| Avantage | Quand c'est important |
|---|---|
| **Incertitude propre** | Intervalles de credibilite interpretables ("90% de chance que θ ∈ [a, b]") |
| **Petits datasets** | Avec un prior informatif, marche la ou sklearn galere |
| **Hierarchie naturelle** | Partial pooling : "estimer chaque ville en empruntant aux autres" |
| **A/B test propre** | "P(B > A) = 92%" — directement interpretable |
| **Mise a jour incrementale** | Le posterior d'aujourd'hui = prior de demain |
| **Regularisation naturelle** | Le prior agit comme une regularisation |

**Concepts cles** :
| Notion | Formule / sens |
|---|---|
| **Prior** `p(θ)` | Croyance a priori sur le parametre |
| **Likelihood** `p(D | θ)` | Probabilite des donnees etant donne θ |
| **Posterior** `p(θ | D) ∝ p(D | θ) × p(θ)` | Croyance mise a jour avec D |
| **Evidence** `p(D)` | Normalisation (souvent intractable) |
| **Conjugate priors** | (prior, likelihood) donnant un posterior analytique (Beta+Bernoulli, Normal+Normal, ...) |
| **MCMC** | Sampler le posterior numeriquement (NUTS, HMC, Metropolis-Hastings) |
| **Credible interval** | Intervalle [a, b] tel que P(θ ∈ [a,b] | D) = 95% |

Versions : `scipy >= 1.11`, code PyMC en commentaire (necessite compile C la premiere fois).

## Plan

1. Setup et concepts (Bayes en une formule)
2. **Cas conjugue Beta-Binomial** : taux de conversion d'un A/B test
3. **A/B test bayesien** : "P(B > A) = ?"
4. **Cas conjugue Normal-Normal** : moyenne d'une variable continue
5. **MCMC (Metropolis-Hastings) from-scratch**
6. **PyMC** : regression bayesienne (presentation + code)
7. **Modele hierarchique** (partial pooling)
8. **Diagnostics MCMC** (R-hat, ESS, trace plots)
9. Pieges et anti-patterns
10. References


## 1. Installation et concepts

### Theoreme de Bayes
`P(θ | D) = P(D | θ) × P(θ) / P(D)`

- Le numerateur est facile (prior × likelihood)
- Le denominateur (`P(D) = ∫ P(D | θ) P(θ) dθ`) est generalement intractable
- → soit **conjugue** (closed-form), soit **MCMC** (sampling), soit **VI** (approximation)


In [ ]:
# pip install -q scipy numpy pandas matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
plt.rcParams["figure.figsize"] = (10, 5)

## 2. Cas conjugue Beta-Binomial — taux de conversion

**Probleme** : on a un bouton sur un site. Sur N visiteurs, k ont clique. Quel est le **vrai** taux de conversion θ ?

**Modele bayesien** :
- **Prior** : `θ ~ Beta(α, β)` (α et β sont nos "essais virtuels" a priori)
- **Likelihood** : `k | θ ~ Binomial(N, θ)`
- **Posterior** : `θ | k, N ~ Beta(α + k, β + N - k)` (conjugaison)

Le posterior est **Beta encore** → on peut le manipuler en closed-form.


In [ ]:
# === Scenario : 1000 visiteurs, 87 clics ===
N, k = 1000, 87
prior_alpha, prior_beta = 1, 1   # Beta(1, 1) = uniforme = ignorance totale

# Posterior
post_alpha = prior_alpha + k
post_beta = prior_beta + N - k

# Distribution posterior
theta_grid = np.linspace(0, 0.2, 1000)
prior_pdf = stats.beta(prior_alpha, prior_beta).pdf(theta_grid)
post_pdf = stats.beta(post_alpha, post_beta).pdf(theta_grid)

fig, ax = plt.subplots()
ax.plot(theta_grid, prior_pdf, label=f"Prior Beta({prior_alpha}, {prior_beta})", linestyle="--")
ax.plot(theta_grid, post_pdf, label=f"Posterior Beta({post_alpha}, {post_beta})", linewidth=2)
ax.axvline(k/N, color="red", linestyle=":", label=f"MLE = k/N = {k/N:.4f}")
ax.set(xlabel="theta (taux de conversion)", ylabel="density",
       title=f"Beta-Binomial update : prior + data ({k}/{N}) -> posterior")
ax.legend()
plt.show()

# Statistiques du posterior
post_dist = stats.beta(post_alpha, post_beta)
print(f"Posterior mean    : {post_dist.mean():.4f}")
print(f"Posterior median  : {post_dist.median():.4f}")
print(f"Posterior std     : {post_dist.std():.4f}")
print(f"95% credible interval : [{post_dist.ppf(0.025):.4f}, {post_dist.ppf(0.975):.4f}]")

**Lecture du resultat** : on n'a pas une **estimation ponctuelle** comme en frequentiste (`MLE = 0.087`), mais **toute la distribution** de θ. On sait que θ est probablement dans [0.070, 0.106] a 95% de confiance.


## 3. A/B test bayesien — "P(B > A) = ?"

**Avantage majeur** : repondre directement a la question business : "Quelle est la probabilite que la variante B soit meilleure que A ?"


In [ ]:
# === Scenario A/B test ===
n_a, k_a = 1000, 87       # Variante A : 87 clics sur 1000
n_b, k_b = 1000, 105      # Variante B : 105 clics sur 1000

# Posteriors (prior Beta(1,1) uniforme)
post_a = stats.beta(1 + k_a, 1 + n_a - k_a)
post_b = stats.beta(1 + k_b, 1 + n_b - k_b)

# Sampling Monte Carlo pour calculer P(B > A) et lift relatif
np.random.seed(SEED)
n_samples = 100000
samples_a = post_a.rvs(n_samples)
samples_b = post_b.rvs(n_samples)

prob_b_better = (samples_b > samples_a).mean()
lift = (samples_b - samples_a) / samples_a
expected_lift = lift.mean()
ci_lift = np.percentile(lift, [2.5, 97.5])

print(f"P(B > A)          : {prob_b_better:.4f}  ({prob_b_better*100:.1f}%)")
print(f"Lift moyen        : {expected_lift*100:+.1f}%")
print(f"95% CI lift       : [{ci_lift[0]*100:+.1f}%, {ci_lift[1]*100:+.1f}%]")

# Decision : si P(B > A) > 95% ET CI lift > 0, on lance B
if prob_b_better > 0.95 and ci_lift[0] > 0:
    print(f"\n→ DECISION : lancer B (forte preuve d'amelioration)")
elif prob_b_better < 0.5:
    print(f"\n→ DECISION : garder A")
else:
    print(f"\n→ DECISION : continuer le test (incertitude trop grande)")

In [ ]:
# Visualisation des posteriors
fig, ax = plt.subplots()
theta_grid = np.linspace(0.05, 0.15, 1000)
ax.fill_between(theta_grid, 0, post_a.pdf(theta_grid), alpha=0.4, label=f"A ({k_a}/{n_a})")
ax.fill_between(theta_grid, 0, post_b.pdf(theta_grid), alpha=0.4, label=f"B ({k_b}/{n_b})", color="C1")
ax.set(xlabel="theta", ylabel="density", title=f"Posteriors A vs B  |  P(B > A) = {prob_b_better:.3f}")
ax.legend()
plt.show()

**Comparaison avec le test frequentiste (Z-test)** :


In [ ]:
# Test frequentiste classique (proportion z-test)
from scipy.stats import norm

p_pool = (k_a + k_b) / (n_a + n_b)
se = np.sqrt(p_pool * (1 - p_pool) * (1/n_a + 1/n_b))
z = (k_b/n_b - k_a/n_a) / se
p_value = 2 * (1 - norm.cdf(abs(z)))

print(f"=== Comparaison interpretation ===")
print(f"Bayesien : P(B > A) = {prob_b_better:.3f}  → interpretation directe")
print(f"Frequentiste : p-value = {p_value:.4f}  → 'probabilite d'observer ces donnees si A=B' (peu intuitif)")

## 4. Cas conjugue Normal-Normal — moyenne d'une mesure

**Probleme** : on mesure une variable continue (taille, temperature, latence d'API). On suppose `X ~ N(μ, σ²)` avec **σ connu**. Quel est μ ?

- **Prior** : `μ ~ N(μ₀, τ₀²)`
- **Likelihood** : `X_i | μ ~ N(μ, σ²)` (N observations)
- **Posterior** : `μ | X ~ N(μ_n, τ_n²)` avec :
  - `1/τ_n² = 1/τ₀² + N/σ²`
  - `μ_n = τ_n² × (μ₀/τ₀² + N·X̄/σ²)`


In [ ]:
# === Latence d'API : on a 50 mesures, connue σ = 30ms ===
np.random.seed(SEED)
true_mu = 250
sigma = 30
N = 50
data = np.random.normal(true_mu, sigma, N)
x_bar = data.mean()

# Prior : on pense que la latence est autour de 200ms, sd 50ms
mu_0, tau_0 = 200, 50

# Posterior
tau_n_sq = 1 / (1 / tau_0**2 + N / sigma**2)
mu_n = tau_n_sq * (mu_0 / tau_0**2 + N * x_bar / sigma**2)
tau_n = np.sqrt(tau_n_sq)

print(f"Donnees : N={N}, X_bar = {x_bar:.2f}, vrai mu = {true_mu}")
print(f"Prior   : N(mu_0={mu_0}, tau_0={tau_0})")
print(f"Post    : N(mu_n={mu_n:.2f}, tau_n={tau_n:.2f})")
print(f"95% CI  : [{mu_n - 1.96*tau_n:.2f}, {mu_n + 1.96*tau_n:.2f}]")

# Visualisation
fig, ax = plt.subplots()
grid = np.linspace(150, 300, 1000)
ax.plot(grid, stats.norm.pdf(grid, mu_0, tau_0), label=f"Prior N({mu_0}, {tau_0})", linestyle="--")
ax.plot(grid, stats.norm.pdf(grid, mu_n, tau_n), label=f"Posterior N({mu_n:.1f}, {tau_n:.2f})", linewidth=2)
ax.axvline(true_mu, color="red", linestyle=":", label=f"Vrai mu = {true_mu}")
ax.axvline(x_bar, color="orange", linestyle=":", label=f"MLE = {x_bar:.2f}")
ax.set(xlabel="mu", ylabel="density", title="Update Normal-Normal")
ax.legend()
plt.show()

## 5. MCMC (Metropolis-Hastings) from-scratch

Quand le posterior n'est pas conjugue, on l'**echantillonne** via MCMC. Algorithme **Metropolis-Hastings** :
1. Etat courant `θ_t`
2. Proposer `θ' ~ q(θ_t)` (ex: gaussienne autour de θ_t)
3. Calculer `r = π(θ') q(θ_t | θ') / (π(θ_t) q(θ' | θ_t))` ou `π` = posterior non-normalise
4. Accepter avec proba `min(1, r)`, sinon rester sur `θ_t`


In [ ]:
# === Posterior non-normalise : Beta(2, 5) (pour la demo, on connait la verite) ===
def log_posterior(theta):
    """log(p(theta | D)) - log(constante)."""
    if theta <= 0 or theta >= 1:
        return -np.inf
    return stats.beta(2, 5).logpdf(theta)

def metropolis_hastings(log_post, n_samples=10_000, proposal_sd=0.05, start=0.5, seed=42):
    rng = np.random.default_rng(seed)
    chain = np.zeros(n_samples)
    chain[0] = start
    n_accepted = 0
    for t in range(1, n_samples):
        current = chain[t - 1]
        proposal = current + rng.normal(0, proposal_sd)
        log_r = log_post(proposal) - log_post(current)
        if np.log(rng.random()) < log_r:
            chain[t] = proposal
            n_accepted += 1
        else:
            chain[t] = current
    return chain, n_accepted / n_samples

chain, acceptance_rate = metropolis_hastings(log_posterior, n_samples=20_000)
print(f"Acceptance rate : {acceptance_rate:.2%}  (cible : ~0.2-0.5)")
print(f"Sample mean     : {chain[2000:].mean():.4f}  (vrai : 2/(2+5) = {2/(2+5):.4f})")
print(f"Sample std      : {chain[2000:].std():.4f}")

In [ ]:
# Visualiser : trace + posterior estime vs vrai
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(chain[:2000], lw=0.5)
axes[0].set(xlabel="iteration", ylabel="theta", title="Trace plot (premieres 2000)")
axes[0].axhline(2/(2+5), color="red", linestyle="--", label=f"Vrai E[theta]")
axes[0].legend()

# Histogram apres burn-in
burn_in = 2000
axes[1].hist(chain[burn_in:], bins=50, density=True, alpha=0.6, label="MCMC samples")
grid = np.linspace(0, 1, 200)
axes[1].plot(grid, stats.beta(2, 5).pdf(grid), "r-", lw=2, label="True Beta(2, 5)")
axes[1].set(xlabel="theta", ylabel="density", title="Posterior estime vs vrai")
axes[1].legend()
plt.tight_layout()
plt.show()

**Notes** :
- Le **burn-in** (premieres ~2000 iterations) est jete : le sampler doit converger d'abord.
- L'**acceptance rate** ideale est 23-44% pour MH classique (Roberts-Rosenthal-Gelman 1997).
- Pour des cas reels en haute dimension : utiliser **NUTS** (No-U-Turn Sampler, default dans PyMC/Stan).


## 6. PyMC — regression bayesienne (code pret a executer)

PyMC est le **standard Python** pour la modelisation bayesienne. Backend PyTensor (Theano fork) qui compile en C la premiere fois.

```python
# pip install -q pymc arviz
import pymc as pm
import arviz as az

# === Regression lineaire bayesienne : Y = a + b*X + eps ===
np.random.seed(42)
N = 100
X = np.random.randn(N)
Y = 2.5 + 1.3 * X + np.random.randn(N) * 0.5

with pm.Model() as model:
    # Priors (faiblement informatifs)
    a = pm.Normal("a", mu=0, sigma=10)
    b = pm.Normal("b", mu=0, sigma=10)
    sigma = pm.HalfNormal("sigma", sigma=1)

    # Likelihood
    mu = a + b * X
    likelihood = pm.Normal("Y_obs", mu=mu, sigma=sigma, observed=Y)

    # Sample posterior (NUTS = No-U-Turn Sampler, par defaut)
    trace = pm.sample(
        draws=2000,
        tune=1000,                # burn-in
        chains=4,                 # 4 chaines parralleles pour R-hat
        target_accept=0.9,
        return_inferencedata=True,
        progressbar=False,
    )

# Resume du posterior
print(az.summary(trace, var_names=["a", "b", "sigma"]))
#       mean   sd  hdi_3%  hdi_97%  mcse_mean  ess_bulk  r_hat
# a    2.498 0.05   2.401    2.594      0.001    8421.0    1.0
# b    1.306 0.05   1.213    1.402      0.001    8311.0    1.0
# sigma 0.49 0.04   0.428    0.567      0.000    7889.0    1.0

# R-hat < 1.01 = convergence OK. ESS > 400 = sampling de qualite.
```

### Diagnostics ArviZ
```python
az.plot_trace(trace, var_names=["a", "b"])   # traces des chaines
az.plot_posterior(trace, var_names=["a", "b"])   # posterior + HDI
az.plot_pair(trace, var_names=["a", "b"])    # correlations posterior
```


## 7. Modele hierarchique — partial pooling

**Cas** : on a N ecoles, on veut estimer le score moyen par ecole.
- **No pooling** : chaque ecole independante → overfit sur les petites ecoles
- **Complete pooling** : tout le monde pareil → ignore les vraies differences
- **Partial pooling** (bayesien hierarchique) : juste milieu

L'idee : les `alpha_ecole` proviennent eux-memes d'une distribution `N(mu_alpha, sigma_alpha)`. Les ecoles avec peu de donnees "empruntent" l'information aux autres via le **shrinkage** vers la moyenne globale.

```python
import pymc as pm

# Donnees : (school_id, student_score)
school_id = np.array([0, 0, 0, 1, 1, 2, 2, 2, 2, 2])
y = np.array([85, 88, 92, 70, 72, 95, 91, 89, 93, 90])
n_schools = 3

with pm.Model() as hierarchical:
    # Hyperparametres (population level)
    mu_alpha = pm.Normal("mu_alpha", mu=80, sigma=20)
    sigma_alpha = pm.HalfNormal("sigma_alpha", sigma=10)

    # Effets par ecole (non-centered parameterization, plus stable)
    alpha_raw = pm.Normal("alpha_raw", mu=0, sigma=1, shape=n_schools)
    alpha = pm.Deterministic("alpha", mu_alpha + sigma_alpha * alpha_raw)

    # Likelihood
    sigma_y = pm.HalfNormal("sigma_y", sigma=5)
    y_obs = pm.Normal("y_obs", mu=alpha[school_id], sigma=sigma_y, observed=y)

    trace = pm.sample(2000, tune=1000, target_accept=0.95)

# Resultat : alpha[ecole_2] sera tire vers la moyenne globale (shrinkage),
# evitant d'overfit sur ses 5 valeurs seules.
```


## 8. Diagnostics MCMC — checklist obligatoire

Avant de croire au resultat d'une MCMC, **toujours** verifier :

| Diagnostic | Bon si | Mauvais si | Cause / Fix |
|---|---|---|---|
| **R-hat** (`r_hat`) | < 1.01 | > 1.05 | Chains pas convergees → +tune, target_accept=0.95 |
| **ESS** (Effective Sample Size, `ess_bulk`) | > 400 | < 100 | Autocorrelation, +draws |
| **Divergences** (NUTS) | 0 | > 10 | Reparametrer (non-centered), target_accept=0.99 |
| **Trace plot** | "comme un chat coiffe" | bornes / drift | Augmenter sampling, verifier prior |
| **PPC** (Posterior Predictive Check) | Replications proches des donnees | Diverge | Modele mal specifie |

### Code diagnostics typique
```python
print(az.summary(trace))                  # tableau R-hat + ESS
az.plot_trace(trace, var_names=["a"])     # visual : trace chaque chaine
az.plot_energy(trace)                     # NUTS energy → divergences
az.plot_forest(trace, var_names=["alpha"])   # comparer parametres

# Posterior Predictive Check
with model:
    ppc = pm.sample_posterior_predictive(trace)
az.plot_ppc(ppc)  # est-ce que le modele reproduit les donnees ?
```


## 9. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correction |
|---|---|
| Prior trop tight (informatif fort) | Influence indument le posterior — utiliser **prior predictive check** |
| Ignorer les divergences NUTS | Reparametrer (non-centered), augmenter `target_accept` |
| 1 seule chaine | Ne peut pas verifier R-hat — utiliser >= 2 chaines (4 ideal) |
| Bayes hierarchique sans non-centered | Souvent divergences → `mu + sigma * raw_norm` au lieu de `Normal(mu, sigma)` direct |
| Confondre intervalle de credibilite (Bayes) et de confiance (frequentiste) | CI bayesien : "θ a 95% de chance d'etre dans [a,b]". CI frequentiste : "[a,b] couvre la vraie valeur 95% des fois si on repete l'experience" (different !) |
| Pas de PPC (Posterior Predictive Check) | Toujours verifier : le modele genere-t-il des donnees realistes ? |
| Trop d'iterations sans verifier ESS | ESS faible = chains correlees, plus d'iterations ne suffisent pas |
| Comparer Bayes Factor sans contexte | Sensible aux priors, BF >> p-value mais interpretation subtile |


## 10. References

### Livres recommandes
- **McElreath (2020)**. *Statistical Rethinking* (2e ed). LE meilleur livre pedagogique. Code R + PyMC.
- **Gelman et al. (2013)**. *Bayesian Data Analysis* (3e ed). Reference academique.
- **Davidson-Pilon**. *Bayesian Methods for Hackers*. **Gratuit en ligne** : https://github.com/CamDavidsonPilon/Probabilistic-Programming-and-Bayesian-Methods-for-Hackers
- **Martin (2018)**. *Bayesian Analysis with Python* (2e ed). Practical, PyMC focus.

### Documentation officielle
- **PyMC** : https://www.pymc.io/
- **NumPyro** (JAX, ultra-rapide) : https://num.pyro.ai/
- **Stan** (la reference inter-langage) : https://mc-stan.org/
- **ArviZ** (diagnostics inter-frameworks) : https://www.arviz.org/
- **Bambi** (formula-based, "GLMs in 1 line") : https://bambinos.github.io/bambi/
- **Edward / TensorFlow Probability** : https://www.tensorflow.org/probability

### Papers fondateurs
- Hoffman & Gelman (2014). *The No-U-Turn Sampler* (NUTS). Default sampler PyMC/Stan.
- Betancourt (2017). *A Conceptual Introduction to Hamiltonian Monte Carlo*. Tres pedagogique.
- Carpenter et al. (2017). *Stan: A Probabilistic Programming Language*.

### Voir aussi (collection)
- [DS_Causal_Inference.ipynb](./DS_Causal_Inference.ipynb) — inference causale (lien fort avec Bayes)
- [DS_Metrics_Reference.ipynb](./DS_Metrics_Reference.ipynb) — section probability metrics (Brier, log loss)
- [ML_Apprentissage_par_Renforcement.ipynb](./ML_Apprentissage_par_Renforcement.ipynb) — Thompson Sampling = bandit bayesien
